<a href="https://colab.research.google.com/github/Sakhiur2022/Signal/blob/main/notebooks/04_modeling/01_class_imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPR-02 — Class Imbalance Fix (SMOTE / ADASYN / Oversampling Comparison)

**Ticket:** SPR-02
**Owner:** Prithila & Sakhiur
**Depends on:** SPR-01 (y-data profiling)
**Folder:** `notebooks/04_modeling/01_class_imbalance.ipynb`
**Output:** resampled train set saved to `data/interim/`, written recommendation on which resampler to use downstream

## 1. Load final dataset

In [ ]:
! wget https://huggingface.co/datasets/Sakhiur/signal/resolve/main/final_dataset_preprocessed_distributed_social_class.csv

In [1]:
from google.colab import drive
import joblib

# Mount Drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
import random
from collections import Counter

np.random.seed(42)
random.seed(42)

df = pd.read_csv('final_dataset_preprocessed_distributed_social_class.csv')

TARGET_COL = 'income_class'

print(df.shape)
df.head()

(751622, 26)


,age,education_years,weekly_workhours,digital_index,social_participation_index,gender_male,disabled,social_participant,agri_worker,disability_severity_ord,...,no_disability,digital_status,workhours_missing,income_class,region_Bali_Nusa,region_Eastern,region_Java,region_Kalimantan,region_Sulawesi,region_Sumatra
0,78.0,6.0,0.0,0.0,0.333333,1,0,1,0,0,...,1,0,1,No_income,0,0,0,0,0,1
1,65.0,6.0,0.0,0.0,0.333333,0,0,1,0,0,...,1,0,1,No_income,0,0,0,0,0,1
2,82.0,6.0,0.0,0.0,0.000000,1,0,0,0,0,...,1,0,1,No_income,0,0,0,0,0,1
3,65.0,6.0,0.0,0.0,0.000000,0,0,0,0,0,...,1,0,1,No_income,0,0,0,0,0,1
4,31.0,12.0,35.0,1.0,0.333333,1,0,1,0,0,...,1,2,0,Lower,0,0,0,0,0,1


## 2. Check current class distribution
This is the number that decides whether you need resampling at all, and how aggressive it should be. Run this before anything else.

In [3]:
class_counts = df[TARGET_COL].value_counts()
print(class_counts)
print()
print('Class proportions:')
print(df[TARGET_COL].value_counts(normalize=True).round(3))

income_class
No_income    327963
Lower        239425
Middle       175908
Upper          8326
Name: count, dtype: int64

Class proportions:
income_class
No_income    0.436
Lower        0.319
Middle       0.234
Upper        0.011
Name: proportion, dtype: float64


## 3. Train/test split
Split BEFORE resampling. Resampling only happens on the training set, never on the test set, or your evaluation numbers become meaningless.

In [16]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train class distribution:')
print(Counter(y_train))
print('Test class distribution:')
print(Counter(y_test))

Train class distribution:
Counter({'No_income': 262370, 'Lower': 191540, 'Middle': 140726, 'Upper': 6661})
Test class distribution:
Counter({'No_income': 65593, 'Lower': 47885, 'Middle': 35182, 'Upper': 1665})


## 4. Baseline: class weights (no resampling)
This is your control condition. Class weights adjust the loss function instead of duplicating/removing rows, and it's worth comparing against, since resampling on a real distribution can distort the problem (flagged as a key learning in the project already).

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print('Computed class weights (for reference in SPR-03 modeling):')
print(class_weight_dict)

Computed class weights (for reference in SPR-03 modeling):
{'Lower': np.float64(0.7848190978385716), 'Middle': np.float64(1.0682052357062661), 'No_income': np.float64(0.5729475549796089), 'Upper': np.float64(22.567820147125055)}


## 5. SMOTE oversampling
Multi-class SMOTE works out of the box in `imblearn`, no changes needed versus the binary case in the SOTA notebook.

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('After SMOTE:')
print(Counter(y_train_smote))

After SMOTE:
Counter({'No_income': 262370, 'Middle': 262370, 'Lower': 262370, 'Upper': 262370})


## 6. ADASYN oversampling
ADASYN focuses more oversampling on harder-to-learn minority examples near the decision boundary, unlike SMOTE which oversamples uniformly. Worth comparing directly.

In [ ]:
from imblearn.over_sampling import ADASYN

adasyn = ADASYN(n_neighbors=3,random_state=42)
try:
    X_train_adasyn, y_train_adasyn = adasyn.fit_resample(X_train, y_train)
    print('After ADASYN:')
    print(Counter(y_train_adasyn))
except ValueError as e:
    # ADASYN can fail if a minority class has too few neighbors
    print('ADASYN failed, likely due to a very small class. Error:')
    print(e)

## 7. SMOTE + Tomek Links
SMOTE oversamples, then Tomek Links removes noisy/borderline pairs from the oversampled set, which tends to produce a cleaner decision boundary than SMOTE alone.

In [ ]:
from imblearn.combine import SMOTETomek

smote_tomek = SMOTETomek(random_state=42)
X_train_st, y_train_st = smote_tomek.fit_resample(X_train, y_train)

print('After SMOTE + Tomek Links:')
print(Counter(y_train_st))

After SMOTE + Tomek Links:
Counter({'Upper': 261982, 'No_income': 259337, 'Middle': 258178, 'Lower': 256395})


## 8. Plain random oversampling (simplest baseline)

In [9]:
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train, y_train)

print('After random oversampling:')
print(Counter(y_train_ros))

After random oversampling:
Counter({'No_income': 262370, 'Middle': 262370, 'Lower': 262370, 'Upper': 262370})


##Load saved variables

In [ ]:
# Load all variables
from google.colab import drive
import joblib
import os


(
    X_train_smote,
    y_train_smote,
    X_train_ros,
    y_train_ros,
    X_train_adasyn,
    y_train_adasyn,
    X_train_st,
    y_train_st
) = joblib.load('/content/drive/MyDrive/ML_Project/saved_variables.pkl')

print("Variables loaded successfully!")

print("Variables loaded successfully!")

## 9. Quick comparison: same simple model on each resampling strategy
Not the final SPR-03 modeling step, just a fast signal for which resampler to recommend. Uses a small depth-capped RF to stay within Colab free-tier limits.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

strategies = {
    'baseline_class_weight': (X_train, y_train, class_weight_dict),
    'smote': (X_train_smote, y_train_smote, None),
    'smote_tomek': (X_train_st, y_train_st, None),
    'random_oversample': (X_train_ros, y_train_ros, None),
}

# add adasyn only if it succeeded above
try:
    strategies['adasyn'] = (X_train_adasyn, y_train_adasyn, None)
except NameError:
    pass

results = {}
for name, (Xtr, ytr, cw) in strategies.items():
    clf = RandomForestClassifier(
        max_depth=10, n_estimators=100, random_state=42,
        class_weight=cw
    )
    clf.fit(Xtr, ytr)
    y_pred = clf.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    results[name] = macro_f1
    print(f'--- {name} ---')
    print(f'Macro F1: {macro_f1:.4f}')
    print(classification_report(y_test, y_pred, digits=3))
    print()

print('Summary (macro F1, higher is better):')
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f'{name}: {score:.4f}')

NameError: name 'X_train_smote' is not defined

## 10. Save the chosen resampled train set
Pick the winner from the comparison above, save it, and this becomes the input for SPR-03 (model training). Update `CHOSEN_STRATEGY` once you've reviewed the results.

In [ ]:
CHOSEN_STRATEGY = 'smote_tomek'  # TODO: set based on the comparison above

chosen_X, chosen_y = strategies[CHOSEN_STRATEGY][0], strategies[CHOSEN_STRATEGY][1]

chosen_X.to_csv('data/interim/X_train_resampled.csv', index=False)
pd.Series(chosen_y, name=TARGET_COL).to_csv('data/interim/y_train_resampled.csv', index=False)
X_test.to_csv('data/interim/X_test.csv', index=False)
y_test.to_csv('data/interim/y_test.csv', index=False)

print(f'Saved resampled train set using: {CHOSEN_STRATEGY}')
print(f'Train shape: {chosen_X.shape}, Test shape: {X_test.shape}')

## 11. Written recommendation
_Fill this in after running the comparison. This is the deliverable SPR-03 depends on._

- Chosen strategy:
- Why:
- Classes that were hardest to separate:
- Any caveats for Sakhiur going into SPR-03:

##Saving varaibles in drive

In [ ]:
joblib.dump(
    (

        X_train_st,
        y_train_st,
        class_weight_dict,
        X_train,
        y_train

    ),
    f'{save_dir}/saved_variables.pkl'
)

print("Variables saved successfully!")

Variables saved successfully!


In [6]:
saved=joblib.load('/content/drive/MyDrive/ML_Project/saved_variables.pkl')

print(type(saved))
print(len(saved))

<class 'tuple'>
5


In [7]:
import joblib

path = '/content/drive/MyDrive/ML_Project/saved_variables.pkl'

# Load what was already saved
old = joblib.load(path)

(
        X_train_st,
        y_train_st,
        class_weight_dict,
        X_train,
        y_train
) = old

# Now save old + new variables together
saved = {
    'X_train_st': X_train_st,
    'y_train_st': y_train_st,
    'class_weight_dict': class_weight_dict,
    'X_train': X_train,
    'y_train': y_train,

    # Add your NEW variables here
    'X_train_smote': X_train_smote,
    'y_train_smote': y_train_smote
}

joblib.dump(saved, path)

print("Old + new variables saved!")

Old + new variables saved!


In [8]:
saved=joblib.load('/content/drive/MyDrive/ML_Project/saved_variables.pkl')

print(type(saved))
print(len(saved))

<class 'dict'>
7


In [11]:
import joblib

path = '/content/drive/MyDrive/ML_Project/saved_variables.pkl'

# Load what was already saved
old = joblib.load(path)

(
        X_train_st,
        y_train_st,
        class_weight_dict,
        X_train,
        y_train,
        X_train_smote,
        y_train_smote
) = old

# Now save old + new variables together
saved = {
    'X_train_st': X_train_st,
    'y_train_st': y_train_st,
    'class_weight_dict': class_weight_dict,
    'X_train': X_train,
    'y_train': y_train,
    'X_train_smote': X_train_smote,
    'y_train_smote': y_train_smote,

    # Add your NEW variables here
    'X_train_ros': X_train_ros,
    'y_train_ros': y_train_ros
}

joblib.dump(saved, path)

print("Old + new variables saved!")

Old + new variables saved!


In [12]:
saved=joblib.load('/content/drive/MyDrive/ML_Project/saved_variables.pkl')

print(type(saved))
print(len(saved))

<class 'dict'>
9


In [ ]:
import joblib

path = '/content/drive/MyDrive/ML_Project/saved_variables.pkl'

# Load what was already saved
old = joblib.load(path)

(
        X_train_st,
        y_train_st,
        class_weight_dict,
        X_train,
        y_train,
        X_train_smote,
        y_train_smote,
        X_train_ros,
        y_train_ros

) = old

# Now save old + new variables together
saved = {
    'X_train_st': X_train_st,
    'y_train_st': y_train_st,
    'class_weight_dict': class_weight_dict,
    'X_train': X_train,
    'y_train': y_train,
    'X_train_smote': X_train_smote,
    'y_train_smote': y_train_smote,
    'X_train_ros': X_train_ros,
    'y_train_ros': y_train_ros,

    # Add your NEW variables here
    'X_train_adasyn': X_train_adasyn,
    'y_train_adasyn': y_train_adasyn

}

joblib.dump(saved, path)

print("Old + new variables saved!")

In [ ]:
saved=joblib.load('/content/drive/MyDrive/ML_Project/saved_variables.pkl')

print(type(saved))
print(len(saved))